In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")
from scipy import stats

In [ ]:
insurance_claims = pd.read_parquet("../data/clean_insurance_claims.parquet")

In [ ]:
# roughly 25:75 split, Y:N
insurance_claims["fraud_reported"].value_counts()

In [ ]:
# remove columns that are not useful for analysis or modelling
insurance_claims = insurance_claims.drop(columns = ["policy_number", "insured_zip", "incident_location"])

In [ ]:
# split data into numeric, categorical, and dates 
numeric_df = insurance_claims.select_dtypes(include = ["int64", "float64"])
categorical_df = insurance_claims.select_dtypes(include = ["object"])
datetime_df = insurance_claims.select_dtypes(include = ["datetime64[ns]"])

numeric_df["fraud_reported"] = insurance_claims["fraud_reported"]
datetime_df["fraud_reported"] = insurance_claims["fraud_reported"]

#### NUMERIC FEATURES
1. The boxplots suggest that most features do not differentiate between fraud and non-fraud. However, noticeable differences are observed for `witnesses`, `total_claim_amount`, `injury_claim`, `property_claim`, and `vehicle_claim`.

2. The histograms show an overlap between fraud and non-fraud, identical to the boxplot observations. The five features from the boxplots show noticeable differences in distributions, with the claim features having a far lower frequency than the lower claims (close to 0).

3. The correlation matrix shows a strong positive correlation between `age` and `months_as_customer`, which is expected as the older customers have been insured longer. Strong correlations are observed between the claim features, which is expected given that the features are derived from one another. There is also a weak positive correlation between `incident_hour_of_the_day` and `number_of_vehicles_involved` with the claim features. There is no correlation with fraud, suggesting that fraud will require multiple features to detect.

4. The independent t-tests comparing fraud vs non-fraud claims revealed that most features did not show statistically significant differences in their means at $\alpha = 0.05$. Only the claim-related variables were statistically significant, suggesting that the size of the claim is associated with fraud.

In [ ]:
numeric_names = numeric_df.select_dtypes(include = ["int64", "float64"]).columns
palette = {"Y": "green", "N": "red"}

fig, axes = plt.subplots(4, 4, figsize = (12, 12), constrained_layout = True)
axes = axes.flatten() 

for ax, column in zip(axes, numeric_names):
    sns.boxplot(data = numeric_df, x = "fraud_reported", y = column, ax = ax, hue = "fraud_reported", palette = palette)
    ax.set_title(column, fontweight = "semibold")

plt.show()

In [ ]:
numeric_names = numeric_df.select_dtypes(include = ["int64", "float64"]).columns

fig, axes = plt.subplots(4, 4, figsize = (12, 12), constrained_layout = True)
axes = axes.flatten() 

for ax, column in zip(axes, numeric_names):
    sns.histplot(data = numeric_df, x = column, hue = "fraud_reported", ax = ax, kde = True, legend = False, palette = palette)
    ax.set_title(column, fontweight = "semibold")

plt.show()

In [ ]:
numeric_df.groupby("fraud_reported")[["witnesses", "total_claim_amount", "injury_claim", "property_claim", "vehicle_claim"]].mean().T

In [ ]:
numeric_df["fraud_numeric"] = numeric_df["fraud_reported"].map({"N": 0, "Y": 1})
numeric_corr = numeric_df.select_dtypes(include = ["int64", "float64"]).corr()

plt.figure(figsize = (16, 10), constrained_layout = True)

sns.heatmap(numeric_corr, annot = True, fmt = ".2g", linewidths = 0.5, cmap = "coolwarm", vmin = -1)
plt.title("Correlation of Numeric Features", fontweight = "semibold")

plt.show()

numeric_df = numeric_df.drop(columns = ["fraud_numeric"])

In [ ]:
t_test_scores = {}

for column in numeric_df:
    if column == "fraud_reported":
        continue
    fraud_column = numeric_df[numeric_df["fraud_reported"] == "Y"][column]
    nonfraud_column = numeric_df[numeric_df["fraud_reported"] == "N"][column]

    t_stat, p_value = stats.ttest_ind(fraud_column, nonfraud_column, equal_var = False)
    t_test_scores[column] = (t_stat, p_value, p_value < 0.01)

t_test_scores_df = pd.DataFrame.from_dict(t_test_scores, orient = "index", columns = ["t_stat", "p_value", "significant"])
t_test_scores_df

#### CATEGORICAL FEATURES
1. The bar chart suggests most features show no difference between fraud and no fraud. However, there are a few features which show a clear difference. In the 'insured_hobbies' feature, 'chess' and 'cross-fit' appear to have an unusually high number of fraud cases compared to non-fraud. In the 'incident_severity' feature, fraud is most commonly reported with 'Major Damage'.

2. The chi-square tests revealed that several categorical variables were significantly associated with fraud at $\alpha = 0.05$. Consistent with the patterns observed in the bar charts, both 'insured_hobbies' and 'incident_severity' are statistically significant. Additional significant features include: 'incident_type', 'collision_type', 'authorities_contacted', 'incident_state', and 'property_damage'.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize = (20, 20), constrained_layout = True)
axes = axes.flatten()

for ax, column in zip(axes, categorical_df.columns):
    sns.countplot(data = categorical_df, x = column, hue = "fraud_reported", ax = ax, palette = palette, legend = False)
    ax.tick_params(axis = "x", rotation = 67.5)
    ax.set_title(column, fontweight = "semibold")

plt.show()

In [ ]:
chi_scores = {}

for column in categorical_df:
    if column == "fraud_reported":
        continue
        
    contigency_table = pd.crosstab(categorical_df[column], categorical_df["fraud_reported"])
    chi_results = stats.chi2_contingency(contigency_table)
    chi_scores[column] = (chi_results.statistic, chi_results.pvalue, chi_results.pvalue < 0.05)

chi_scores_df = pd.DataFrame.from_dict(chi_scores, orient = "index", columns = ["chi_stat", "p_value", "significant"])
chi_scores_df

#### DATETIME FEATURES
1. In the line charts, there is no clear indication of any relationship between days and months for the 'policy_bind_date' and 'incident_date' features across fraud and non-fraud activity.

2. The box plot reveals overlapping distributions between fraud and non-fraud activity. Besides the month for the 'incident_date ', there are slight differences in the medians of the two groups, but there is no clear separation between them that demonstrates a clear relationship for fraud occurrence.

In [ ]:
datetime_df["policy_bind_date_month"] = datetime_df["policy_bind_date"].dt.month
datetime_df["policy_bind_date_day"] = datetime_df["policy_bind_date"].dt.day
datetime_df["incident_date_month"] = datetime_df["incident_date"].dt.month
datetime_df["incident_date_day"] = datetime_df["incident_date"].dt.day

policy_fraud_rate_month = datetime_df.groupby(["policy_bind_date_month", "fraud_reported"]).size().reset_index(name = "count")
policy_fraud_rate_day = datetime_df.groupby(["policy_bind_date_day", "fraud_reported"]).size().reset_index(name = "count")
incident_fraud_rate_month = datetime_df.groupby(["incident_date_month", "fraud_reported"]).size().reset_index(name = "count")
incident_fraud_rate_day = datetime_df.groupby(["incident_date_day", "fraud_reported"]).size().reset_index(name = "count")

data_column = [(policy_fraud_rate_month, "policy_bind_date_month"), (policy_fraud_rate_day, "policy_bind_date_day"), 
               (incident_fraud_rate_month, "incident_date_month"), (incident_fraud_rate_day, "incident_date_day")]

fig, axes = plt.subplots(2, 2, figsize = (10, 5), constrained_layout = True)
axes = axes.flatten()

for ax, (data, column) in zip(axes, data_column):
    sns.lineplot(data = data, x = column, y = "count",  hue = "fraud_reported", palette = palette, ax = ax, legend = False)
    ax.set_title(column, fontweight = "semibold")

plt.show()

In [ ]:
datetime_names = datetime_df.select_dtypes(include = ["int32"]).columns

fig, axes = plt.subplots(2, 2, figsize = (6, 6), constrained_layout = True)
axes = axes.flatten() 

for ax, column in zip(axes, datetime_names):
    sns.boxplot(data = datetime_df, y = column, x = "fraud_reported",  hue = "fraud_reported", palette = palette, ax = ax)
    ax.set_title(column, fontweight = "semibold")

plt.show()